# Probabilistic ensemble forecasting

Load an opinionated four-member Granite ensemble recipe, forecast hourly ETTh1 observations, and compare with held-out actuals. The recipe selects the checkpoints, quantiles, and aggregation settings; TTM's revision remains dynamically selected from context length and horizon.

## Install

For this development branch, install the checkout into the notebook environment. Restart the kernel if the package was already imported. After release, the installation can use the published granite-tsfm[notebooks] package.

In [ ]:
from pathlib import Path

repo_root = next(
    p for p in [Path.cwd(), *Path.cwd().parents]
    if (p / "pyproject.toml").is_file() and (p / "tsfm_public").is_dir()
)

%pip install -q -e "{repo_root}[notebooks]"

In [ ]:
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

from tsfm_public.models.ensemble.configuration_ensemble import ProbabilisticEnsembleConfig
from tsfm_public.models.ensemble.modeling_ensemble import QuantileEnsembleForecaster
from tsfm_public.toolkit.time_series_preprocessor import prepare_data_splits

## Load the recipe

The local directory contains config.json. Once the recipe is published, replace this directory with its Hugging Face repository ID. `QuantileEnsembleForecaster.from_pretrained()` is the minimal path: it loads the recipe and constructs its members. Linear pooling is the default.

| aggregation_method | Behavior | Method-specific options |
|---|---|---|
| `"linear_pool"` (default) | Pools member quantile forecasts; equal influence by default | None |
| `"iqr_weighted"` | Gives greater weight to members with narrower interquantile ranges | `temperature`, `max_weight` in `config.iqr_weighted_options` |

IQR defaults are `temperature=0.5` and `max_weight=0.4`. Temperature must be finite and positive: lower values concentrate weights, while higher values move them toward uniform weighting. The cap must be between `1 / n_members` and `1`, or `None` to disable capping. If you reduce the member count, check that the cap remains valid. Narrower intervals do not necessarily indicate greater accuracy.

For advanced customization, load `ProbabilisticEnsembleConfig` separately, modify it, and pass it to `QuantileEnsembleForecaster.from_config()`. Changes can be made to `config.aggregation_method`, `config.members`, or `config.iqr_weighted_options` and saved with `config.save_pretrained("./my_ensemble_recipe")`. Invalid recipes fail before checkpoints are loaded.

In [ ]:

device = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")

repo_id = "ibm-granite/granite-timeseries-ensemble-r1"

ensemble = QuantileEnsembleForecaster.from_pretrained(
    repo_id,
    device=device,
    token=True,
)

In [ ]:
# Advanced customization (use instead of the previous construction call):

# config = ProbabilisticEnsembleConfig.from_pretrained(
#     repo_id,
#     token=True,
# )
# config.aggregation_method = "iqr_weighted"
# config.iqr_weighted_options["temperature"] = 0.5
# ensemble = QuantileEnsembleForecaster.from_config(config, device=device)

## Prepare a held-out forecast window

Use the same ETTh1 split boundaries as the PatchTST getting-started notebook. Only the context is passed to the models; the next horizon is retained as ground truth. No training is needed. TTM uses the latest history supported by its selected checkpoint while preserving the common cutoff.

In [ ]:
context_length = 4096
prediction_length = 96
target_column = "HUFL"

raw_df = pd.read_csv(
    "https://raw.githubusercontent.com/zhouhaoyi/ETDataset/main/ETT-small/ETTh1.csv",
    parse_dates=["date"],
)
df = raw_df[["date", target_column]].rename(columns={"date": "timestamp", target_column: "target"})
_, _, test_df = prepare_data_splits(
    df,
    context_length=context_length,
    split_config={"train": [0, 8640], "valid": [8640, 11520], "test": [11520, 14400]},
)
forecast_input = test_df.iloc[:context_length].reset_index(drop=True)
forecast_actuals = test_df.iloc[context_length:context_length + prediction_length].reset_index(drop=True)

In [ ]:
result = ensemble(
    forecast_input,
    timestamp_column="timestamp",
    target_columns=["target"],
    context_length=context_length,
    prediction_length=prediction_length,
)
# Retain the known actuals and cutoff for this notebook's single forecast window.
result.actuals = forecast_actuals["target"].to_numpy()[None, :, None].tolist()
result.cutoff_dates = [str(forecast_input["timestamp"].iloc[-1])]
print(result.message)

## Plot the forecast

Show recent history, the ensemble median, its 10th–90th percentile interval, and held-out actuals. Narrower intervals do not necessarily indicate greater accuracy; evaluating methods requires multiple held-out windows.

In [ ]:
median = np.asarray(result.predicted)[0, :, 0]
quantiles = np.asarray(result.predicted_quantiles)[0, :, 0, :]
levels = ensemble.quantile_levels
timestamps = forecast_actuals["timestamp"]
history = forecast_input.tail(4 * prediction_length)

plt.style.use("seaborn-v0_8-whitegrid")
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(history["timestamp"], history["target"], label="Context", color="black", linewidth=1)
ax.plot(timestamps, forecast_actuals["target"], label="Actuals", color="blue", linewidth=1)
ax.plot(timestamps, median, label="Ensemble median", color="orange", linestyle="--")
ax.fill_between(
    timestamps,
    quantiles[:, levels.index(0.1)],
    quantiles[:, levels.index(0.9)],
    label="10th–90th percentile interval", color="lightblue", alpha=0.7,
)
ax.axvline(forecast_input["timestamp"].iloc[-1], color="gray", linestyle=":")
ax.set(title=f"{result.metadata['method']}: {target_column}", xlabel="Timestamp", ylabel=target_column)
ax.legend(loc="upper left")
fig.autofmt_xdate()
fig.tight_layout()
plt.show()